In [ ]:
from pyspark.sql import functions as F

catalog = "retail_operations_dev"

customers_raw = spark.table(
    f"{catalog}.bronze.customers_raw"
)

display(customers_raw.orderBy("customer_id"))

In [ ]:
def has_text(column_name):
    return (
        F.length(F.trim(F.col(column_name))) > 0
    )


customer_rules = [
    (
        has_text("customer_id"),
        "Missing customer_id"
    ),
    (
        has_text("customer_name"),
        "Missing customer_name"
    ),
    (
        has_text("email"),
        "Missing email"
    ),
    (
        F.col("country_code").rlike("^[A-Z]{2}$"),
        "Invalid country_code format"
    ),
    (
        F.col("customer_segment").isin(
            "standard", "premium", "business"
        ),
        "Missing or unsupported customer_segment"
    ),
    (
        F.col("is_active").isNotNull(),
        "Missing is_active"
    ),
    (
        F.col("created_at").isNotNull(),
        "Missing created_at"
    ),
    (
        F.col("updated_at").isNotNull(),
        "Missing updated_at"
    ),
    (
        F.col("updated_at") >= F.col("created_at"),
        "updated_at is earlier than created_at"
    ),
]

checked_customers = customers_raw.withColumn(
    "rejection_reasons",
    F.filter(
        F.array(*[
            F.when(
                ~F.coalesce(condition, F.lit(False)),
                F.lit(reason)
            )
            for condition, reason in customer_rules
        ]),
        lambda reason: reason.isNotNull()
    )
).withColumn(
    "is_valid",
    F.size("rejection_reasons") == 0
)

display(
    checked_customers.select(
        "customer_id",
        "customer_segment",
        "is_valid",
        "rejection_reasons"
    ).orderBy("customer_id")
)

In [ ]:
display(
    checked_customers.groupBy("is_valid").count()
)

duplicate_customer_ids = (
    customers_raw
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

display(duplicate_customer_ids)

assert duplicate_customer_ids.limit(1).count() == 0, (
    "Duplicate customer IDs found in customers_raw."
)

In [ ]:
customer_columns = [
    "customer_id",
    "customer_name",
    "email",
    "country_code",
    "customer_segment",
    "is_active",
    "created_at",
    "updated_at",
]

valid_customers = (
    checked_customers
    .filter(F.col("is_valid"))
    .select(*customer_columns)
)

rejected_customers = (
    checked_customers
    .filter(
        ~F.col("is_valid")
    )
)

display(rejected_customers)
assert rejected_customers.limit(1).count() == 0, (
    "Customer validation failed. Review rejected_customers "
    "before updating Silver."
)

In [ ]:
from delta.tables import DeltaTable

silver_customer_table = f"{catalog}.silver.customers_current"

valid_customers.createOrReplaceTempView(
    "valid_customers_snapshot"
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {silver_customer_table}   
    USING DELTA
    AS 
    SELECT * 
    FROM valid_customers_snapshot       
    WHERE 1 = 0
""")

(
    DeltaTable.forName(spark , silver_customer_table)
    .alias('target')
    .merge(
        valid_customers.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .whenNotMatchedBySourceDelete()
    .execute()
)



In [ ]:
saved_customers = (
    spark.table(silver_customer_table)
    .select(*customer_columns)
)

assert(
    valid_customers.exceptAll(saved_customers)
    .limit(1).count() == 0
), "Some source customer rows are missing or different in Silver."

assert(
    saved_customers.exceptAll(valid_customers)
    .limit(1).count() == 0
),  "Silver contains extra or different customer rows."

display(
    saved_customers
    .agg(
        F.count("*").alias("saved_customers"),
        F.countDistinct("customer_id").alias("unique_customers")
    )
)

display(
    saved_customers
    .filter(F.col("customer_id") == "C001")
)